# TFT Model Real-Time Simulation
This notebook loads real-time S&P500 ETF (SPY) data and applies feature engineering to prepare data for the Temporal Fusion Transformer model.

### Download realtime data from Yahoo Finance

In [ ]:
import numpy as np
import pandas as pd
import yfinance as yf
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Download recent SPY data (1-minute interval, max 7 days for free tier)
# For real-time simulation, we need enough history for feature calculation
ticker = 'SPY'
df = yf.download(ticker, period='5d', interval='1m', progress=False)

# Flatten multi-index columns if present
if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.droplevel(1)

# Reset index to have timestamp as column
df = df.reset_index()
df.rename(columns={'Datetime': 'timestamp', 'Open': 'open', 'High': 'high', 
                   'Low': 'low', 'Close': 'close', 'Volume': 'volume'}, inplace=True)

# Calculate VWAP (Volume Weighted Average Price)
df['vwap'] = (df['volume'] * (df['high'] + df['low'] + df['close']) / 3).cumsum() / df['volume'].cumsum()

print(f"Loaded {len(df)} rows of {ticker} data")
print(f"Date range: {df['timestamp'].min()} to {df['timestamp'].max()}")
df.head()

Loaded 1626 rows of SPY data
Date range: 2026-03-09 13:30:00+00:00 to 2026-03-13 14:35:00+00:00


Price,timestamp,close,high,low,open,volume,vwap
0,2026-03-09 13:30:00+00:00,667.450012,667.539978,666.299988,666.390015,3826415,667.096659
1,2026-03-09 13:31:00+00:00,666.640015,667.630005,666.599976,667.440002,497932,667.080540
2,2026-03-09 13:32:00+00:00,666.825012,667.119995,666.539978,666.640015,323956,667.062962
3,2026-03-09 13:33:00+00:00,665.955017,666.929993,665.739990,666.820007,426876,666.991079
4,2026-03-09 13:34:00+00:00,666.349976,666.500000,665.890015,665.950012,241518,666.957263


### Recreate the features as they are in our training data

In [ ]:
def calculate_all_features(df):
    """
    Calculate all 75 features required for the TFT model.
    Input df must have: timestamp, open, high, low, close, volume, vwap
    """
    df = df.copy()
    
    # ========================
    # 1. RETURNS (5 features)
    # ========================
    df['returns_1m'] = df['close'].pct_change(1)
    df['returns_5m'] = df['close'].pct_change(5)
    df['returns_15m'] = df['close'].pct_change(15)
    df['returns_30m'] = df['close'].pct_change(30)
    df['returns_60m'] = df['close'].pct_change(60)
    
    # ========================
    # 2. PRICE RATIOS (6 features)
    # ========================
    df['high_low_ratio'] = df['high'] / df['low']
    df['close_open_ratio'] = df['close'] / df['open']
    df['high_close_ratio'] = df['high'] / df['close']
    df['low_close_ratio'] = df['low'] / df['close']
    df['upper_shadow'] = (df['high'] - np.maximum(df['open'], df['close'])) / (df['high'] - df['low'] + 1e-10)
    df['lower_shadow'] = (np.minimum(df['open'], df['close']) - df['low']) / (df['high'] - df['low'] + 1e-10)
    
    # ========================
    # 3. SMA FEATURES (11 features)
    # ========================
    for period in [5, 10, 20, 30]:
        sma = df['close'].rolling(window=period).mean()
        df[f'sma_{period}_slope'] = sma.pct_change()
        df[f'close_to_sma_{period}'] = (df['close'] - sma) / sma
    
    # SMA 60 - only ratio, no slope in original
    sma_60 = df['close'].rolling(window=60).mean()
    df['close_to_sma_60'] = (df['close'] - sma_60) / sma_60
    
    # SMA 120 - with slope
    sma_120 = df['close'].rolling(window=120).mean()
    df['sma_120_slope'] = sma_120.pct_change()
    df['close_to_sma_120'] = (df['close'] - sma_120) / sma_120
    
    # ========================
    # 4. MACD (2 features)
    # ========================
    ema_12 = df['close'].ewm(span=12, adjust=False).mean()
    ema_26 = df['close'].ewm(span=26, adjust=False).mean()
    df['macd'] = ema_12 - ema_26
    signal_line = df['macd'].ewm(span=9, adjust=False).mean()
    df['macd_histogram'] = df['macd'] - signal_line
    
    # ========================
    # 5. VOLATILITY (4 features)
    # ========================
    df['volatility_5'] = df['returns_1m'].rolling(window=5).std()
    df['volatility_10'] = df['returns_1m'].rolling(window=10).std()
    df['volatility_20'] = df['returns_1m'].rolling(window=20).std()
    df['volatility_60'] = df['returns_1m'].rolling(window=60).std()
    
    # ========================
    # 6. ATR (2 features)
    # ========================
    high_low = df['high'] - df['low']
    high_close = np.abs(df['high'] - df['close'].shift())
    low_close = np.abs(df['low'] - df['close'].shift())
    tr = np.maximum(high_low, np.maximum(high_close, low_close))
    df['atr_14'] = tr.rolling(window=14).mean()
    df['atr_60'] = tr.rolling(window=60).mean()
    
    # ========================
    # 7. BOLLINGER BANDS (4 features)
    # ========================
    for period in [20, 60]:
        sma = df['close'].rolling(window=period).mean()
        std = df['close'].rolling(window=period).std()
        df[f'bb_std_{period}'] = std
        df[f'bb_position_{period}'] = (df['close'] - sma) / (2 * std + 1e-10)
    
    # ========================
    # 8. RSI (3 features)
    # ========================
    def calc_rsi(series, period):
        delta = series.diff()
        gain = delta.where(delta > 0, 0).rolling(window=period).mean()
        loss = (-delta.where(delta < 0, 0)).rolling(window=period).mean()
        rs = gain / (loss + 1e-10)
        return 100 - (100 / (1 + rs))
    
    df['rsi_14'] = calc_rsi(df['close'], 14)
    df['rsi_20'] = calc_rsi(df['close'], 20)
    df['rsi_60'] = calc_rsi(df['close'], 60)
    
    # ========================
    # 9. STOCHASTIC (3 features)
    # ========================
    for period in [14, 60]:
        lowest_low = df['low'].rolling(window=period).min()
        highest_high = df['high'].rolling(window=period).max()
        df[f'stoch_k_{period}'] = 100 * (df['close'] - lowest_low) / (highest_high - lowest_low + 1e-10)
    df['stoch_d_14'] = df['stoch_k_14'].rolling(window=3).mean()
    
    # ========================
    # 10. RATE OF CHANGE (2 features)
    # ========================
    df['roc_10'] = df['close'].pct_change(10) * 100
    df['roc_20'] = df['close'].pct_change(20) * 100
    
    # ========================
    # 11. VOLUME INDICATORS (12 features)
    # ========================
    df['volume_change'] = df['volume'].pct_change(1)
    df['volume_change_5m'] = df['volume'].pct_change(5)
    
    for period in [5, 10, 20, 60]:
        df[f'volume_sma_{period}'] = df['volume'].rolling(window=period).mean()
        df[f'volume_ratio_{period}'] = df['volume'] / (df[f'volume_sma_{period}'] + 1e-10)
    
    # ========================
    # 12. OBV, VPT, MFI (5 features)
    # ========================
    # OBV
    obv = np.where(df['close'] > df['close'].shift(), df['volume'],
                   np.where(df['close'] < df['close'].shift(), -df['volume'], 0))
    df['obv'] = np.cumsum(obv)
    obv_sma = pd.Series(df['obv']).rolling(window=20).mean()
    df['obv_ratio'] = df['obv'] / (obv_sma + 1e-10)
    
    # VPT (Volume Price Trend)
    df['vpt'] = (df['volume'] * df['close'].pct_change()).cumsum()
    
    # MFI (Money Flow Index)
    def calc_mfi(df, period):
        typical_price = (df['high'] + df['low'] + df['close']) / 3
        money_flow = typical_price * df['volume']
        positive_flow = money_flow.where(typical_price > typical_price.shift(), 0).rolling(window=period).sum()
        negative_flow = money_flow.where(typical_price < typical_price.shift(), 0).rolling(window=period).sum()
        mfi = 100 - (100 / (1 + positive_flow / (negative_flow + 1e-10)))
        return mfi
    
    df['mfi_14'] = calc_mfi(df, 14)
    df['mfi_60'] = calc_mfi(df, 60)
    
    # ========================
    # 13. VWAP RATIOS (2 features)
    # ========================
    vwap_20 = (df['volume'] * df['close']).rolling(window=20).sum() / (df['volume'].rolling(window=20).sum() + 1e-10)
    vwap_60 = (df['volume'] * df['close']).rolling(window=60).sum() / (df['volume'].rolling(window=60).sum() + 1e-10)
    df['close_to_vwap_20'] = (df['close'] - vwap_20) / vwap_20
    df['close_to_vwap_60'] = (df['close'] - vwap_60) / vwap_60
    
    # ========================
    # 14. TIME FEATURES (10 features)
    # ========================
    df['hour'] = df['timestamp'].dt.hour
    df['minute'] = df['timestamp'].dt.minute
    df['day_of_week'] = df['timestamp'].dt.dayofweek
    df['is_morning'] = ((df['hour'] >= 9) & (df['hour'] < 12)).astype(int)
    df['is_afternoon'] = ((df['hour'] >= 12) & (df['hour'] < 16)).astype(int)
    
    # Cyclical encoding
    df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
    df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
    df['minute_sin'] = np.sin(2 * np.pi * df['minute'] / 60)
    df['minute_cos'] = np.cos(2 * np.pi * df['minute'] / 60)
    df['day_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 7)
    
    # ========================
    # 15. VOLUME LAGS (5 features)
    # ========================
    df['volume_lag_1'] = df['volume'].shift(1)
    df['volume_lag_5'] = df['volume'].shift(5)
    df['volume_lag_15'] = df['volume'].shift(15)
    df['volume_lag_30'] = df['volume'].shift(30)
    df['volume_lag_60'] = df['volume'].shift(60)
    
    # ========================
    # 16. TARGET VARIABLES (2 features)
    # ========================
    df['target_close_60m'] = df['close'].shift(-60)
    df['target_return_60m'] = df['close'].pct_change(60).shift(-60)
    
    return df

# Apply feature engineering
df_features = calculate_all_features(df)

# Check feature count
feature_cols = ['open', 'high', 'low', 'close', 'volume', 'vwap', 'returns_1m',
       'returns_5m', 'returns_15m', 'returns_30m', 'returns_60m',
       'high_low_ratio', 'close_open_ratio', 'high_close_ratio',
       'low_close_ratio', 'upper_shadow', 'lower_shadow', 'sma_5_slope',
       'close_to_sma_5', 'sma_10_slope', 'close_to_sma_10', 'sma_20_slope',
       'close_to_sma_20', 'sma_30_slope', 'close_to_sma_30', 'close_to_sma_60',
       'sma_120_slope', 'close_to_sma_120', 'macd', 'macd_histogram',
       'volatility_5', 'volatility_10', 'volatility_20', 'volatility_60',
       'atr_14', 'atr_60', 'bb_std_20', 'bb_position_20', 'bb_std_60',
       'bb_position_60', 'rsi_14', 'rsi_20', 'rsi_60', 'stoch_k_14',
       'stoch_d_14', 'stoch_k_60', 'roc_10', 'roc_20', 'volume_change',
       'volume_change_5m', 'volume_sma_5', 'volume_ratio_5', 'volume_sma_10',
       'volume_ratio_10', 'volume_sma_20', 'volume_ratio_20', 'volume_sma_60',
       'volume_ratio_60', 'obv', 'obv_ratio', 'vpt', 'mfi_14', 'mfi_60',
       'close_to_vwap_20', 'close_to_vwap_60', 'hour', 'minute', 'day_of_week',
       'is_morning', 'is_afternoon', 'hour_sin', 'hour_cos', 'minute_sin',
       'minute_cos', 'day_sin', 'volume_lag_1', 'volume_lag_5',
       'volume_lag_15', 'volume_lag_30', 'volume_lag_60', 'target_close_60m',
       'target_return_60m']

print(f"Total features expected: {len(feature_cols)}")
print(f"Features created: {len([c for c in feature_cols if c in df_features.columns])}")
missing = [c for c in feature_cols if c not in df_features.columns]
if missing:
    print(f"Missing features: {missing}")
else:
    print("All 75 features successfully created!")

# Filter to only valid rows (drop NaN from rolling calculations)
df_filtered = df_features[feature_cols].dropna()
print(f"\nRows after filtering NaN: {len(df_filtered)}")
df_filtered.head()

Total features expected: 82
Features created: 82
All 75 features successfully created!

Rows after filtering NaN: 1446


Price,open,high,low,close,volume,vwap,returns_1m,returns_5m,returns_15m,returns_30m,...,minute_sin,minute_cos,day_sin,volume_lag_1,volume_lag_5,volume_lag_15,volume_lag_30,volume_lag_60,target_close_60m,target_return_60m
120,669.770020,669.809998,669.365479,669.469971,141094,665.856154,-0.000448,-0.000090,0.000366,0.002201,...,5.665539e-16,-1.000000,0.0,85253.0,385287.0,200676.0,98694.0,201996.0,670.429993,0.001434
121,669.479980,669.760010,669.429993,669.580017,104630,665.867887,0.000164,-0.000328,0.000717,0.002125,...,-1.045285e-01,-0.994522,0.0,141094.0,221628.0,146508.0,129674.0,369613.0,670.304993,0.001083
122,669.609985,669.619995,669.184998,669.260010,261554,665.895065,-0.000478,-0.000791,0.000254,0.001646,...,-2.079117e-01,-0.978148,0.0,104630.0,186266.0,185970.0,141752.0,131885.0,670.109985,0.001270
123,669.260010,669.260010,669.080017,669.200012,192313,665.913783,-0.000090,-0.000597,0.000733,0.000867,...,-3.090170e-01,-0.951057,0.0,261554.0,180655.0,183420.0,245256.0,233711.0,670.570007,0.002047
124,669.193726,669.299988,668.669983,668.849976,239271,665.935085,-0.000523,-0.001374,-0.000030,0.000509,...,-4.067366e-01,-0.913545,0.0,192313.0,85253.0,229068.0,235464.0,154786.0,670.840027,0.002975


In [ ]:
from pytorch_forecasting import TemporalFusionTransformer

# Loading the trained model
model = TemporalFusionTransformer.load_from_checkpoint('models/tft_checkpoint_latest.ckpt')
model.eval()

print("Model loaded successfully!")
print(f"Model type: {type(model).__name__}")
print(f"Encoder length: {model.hparams.max_encoder_length}")


Model loaded successfully!
Model type: TemporalFusionTransformer
Encoder length: 60


In [7]:
# Cell 4: Prepare data for prediction using TimeSeriesDataSet
from pytorch_forecasting import TimeSeriesDataSet
import torch

# Get model parameters - check available hparams first
print("Available model hparams:")
print(list(model.hparams.keys()))

# Get encoder length from hparams or dataset_parameters
if hasattr(model.hparams, 'max_encoder_length'):
    max_encoder_length = model.hparams.max_encoder_length
elif hasattr(model, 'dataset_parameters') and 'max_encoder_length' in model.dataset_parameters:
    max_encoder_length = model.dataset_parameters['max_encoder_length']
else:
    max_encoder_length = 60  # Default fallback

# Get prediction length - TFT models typically store this
if hasattr(model.hparams, 'max_prediction_length'):
    max_prediction_length = model.hparams.max_prediction_length
elif hasattr(model, 'dataset_parameters') and 'max_prediction_length' in model.dataset_parameters:
    max_prediction_length = model.dataset_parameters['max_prediction_length']
else:
    max_prediction_length = 60  # Default to 60 min prediction horizon

print(f"\nModel encoder length: {max_encoder_length}")
print(f"Model prediction length: {max_prediction_length}")

# Prepare data for TimeSeriesDataSet - need to add time_idx and group
df_pred = df_features.copy()
df_pred = df_pred.dropna(subset=feature_cols[:-2])  # Drop NaN but keep target cols
df_pred = df_pred.reset_index(drop=True)
df_pred['time_idx'] = range(len(df_pred))
df_pred['group'] = 'SPY'  # Single group for one asset

# Define feature categories matching training setup
time_varying_known_reals = ['hour', 'minute', 'day_of_week', 'is_morning', 'is_afternoon',
                            'hour_sin', 'hour_cos', 'minute_sin', 'minute_cos', 'day_sin']

time_varying_unknown_reals = [col for col in feature_cols if col not in time_varying_known_reals 
                              and col not in ['target_close_60m', 'target_return_60m']]

# Use only the last portion needed for prediction
# Need encoder_length + prediction_length rows minimum
min_rows_needed = max_encoder_length + max_prediction_length
df_recent = df_pred.iloc[-(min_rows_needed + 100):].copy()  # Extra buffer
df_recent['time_idx'] = range(len(df_recent))

print(f"\nUsing {len(df_recent)} rows for prediction")
print(f"Time range: {df_recent['timestamp'].iloc[0]} to {df_recent['timestamp'].iloc[-1]}")

Available model hparams:
['hidden_size', 'lstm_layers', 'dropout', 'output_size', 'attention_head_size', 'max_encoder_length', 'static_categoricals', 'static_reals', 'time_varying_categoricals_encoder', 'time_varying_categoricals_decoder', 'categorical_groups', 'time_varying_reals_encoder', 'time_varying_reals_decoder', 'x_reals', 'x_categoricals', 'hidden_continuous_size', 'hidden_continuous_sizes', 'embedding_sizes', 'embedding_paddings', 'embedding_labels', 'learning_rate', 'log_interval', 'log_val_interval', 'log_gradient_flow', 'reduce_on_plateau_patience', 'monotone_constraints', 'share_single_variable_networks', 'causal_attention', 'mask_bias', 'output_transformer', 'dataset_parameters', 'reduce_on_plateau_reduction', 'reduce_on_plateau_min_lr', 'weight_decay', 'optimizer_params', 'optimizer']

Model encoder length: 60
Model prediction length: 60

Using 220 rows for prediction
Time range: 2026-03-12 17:26:00+00:00 to 2026-03-13 14:35:00+00:00


In [11]:
df_recent

Price,timestamp,close,high,low,open,volume,vwap,returns_1m,returns_5m,returns_15m,...,day_sin,volume_lag_1,volume_lag_5,volume_lag_15,volume_lag_30,volume_lag_60,target_close_60m,target_return_60m,time_idx,group
1286,2026-03-12 17:26:00+00:00,669.010010,669.159973,668.750000,669.099976,97925,673.699085,-0.000120,0.000860,-0.000523,...,0.433884,117209.0,149497.0,69434.0,115863.0,218185.0,668.140015,-0.001300,0,SPY
1287,2026-03-12 17:27:00+00:00,668.895020,669.309998,668.890015,669.000000,111631,673.697257,-0.000172,-0.000456,-0.000575,...,0.433884,97925.0,173908.0,67589.0,117386.0,252798.0,667.984985,-0.001361,1,SPY
1288,2026-03-12 17:28:00+00:00,668.520020,668.919983,668.520020,668.900024,118229,673.695166,-0.000561,-0.000822,-0.000852,...,0.433884,111631.0,147849.0,76033.0,87522.0,160431.0,667.830017,-0.001032,2,SPY
1289,2026-03-12 17:29:00+00:00,668.559998,668.669983,668.380005,668.530029,79906,673.693721,0.000060,-0.000717,-0.000207,...,0.433884,118229.0,340913.0,132779.0,80639.0,176661.0,668.190002,-0.000553,3,SPY
1290,2026-03-12 17:30:00+00:00,668.940002,668.969971,668.530029,668.559998,104546,673.691932,0.000568,-0.000224,0.000404,...,0.433884,79906.0,117209.0,149511.0,103434.0,241860.0,668.229919,-0.001062,4,SPY
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1501,2026-03-13 14:31:00+00:00,670.090027,670.109985,669.369995,669.450012,247419,672.856393,0.000934,0.002304,0.002438,...,-0.433884,187898.0,181570.0,197039.0,130072.0,463795.0,NaN,NaN,215,SPY
1502,2026-03-13 14:32:00+00:00,670.200012,670.229980,669.850891,670.070007,128868,672.855344,0.000164,0.002086,0.003369,...,-0.433884,247419.0,281166.0,401961.0,324914.0,361716.0,NaN,NaN,216,SPY
1503,2026-03-13 14:33:00+00:00,669.849976,670.224609,669.760010,670.210022,158101,672.853989,-0.000522,0.001256,0.003100,...,-0.433884,128868.0,164318.0,274974.0,213154.0,378874.0,NaN,NaN,217,SPY
1504,2026-03-13 14:34:00+00:00,669.500000,669.830017,669.169678,669.820129,163095,672.852378,-0.000522,0.000987,0.001976,...,-0.433884,158101.0,140782.0,190369.0,144880.0,352436.0,NaN,NaN,218,SPY


In [13]:
# Cell 5: Inspect model's expected dataset parameters
# The model stores the dataset configuration it was trained with

print("Model's dataset parameters:")
print("=" * 50)

# Get the dataset parameters from the model
if hasattr(model, 'dataset_parameters'):
    dataset_params = model.dataset_parameters
    print(f"max_encoder_length: {dataset_params.get('max_encoder_length')}")
    print(f"max_prediction_length: {dataset_params.get('max_prediction_length')}")
    print(f"target: {dataset_params.get('target')}")
    print(f"group_ids: {dataset_params.get('group_ids')}")
    print(f"\ntime_varying_known_categoricals: {dataset_params.get('time_varying_known_categoricals')}")
    print(f"time_varying_unknown_categoricals: {dataset_params.get('time_varying_unknown_categoricals')}")
    print(f"static_categoricals: {dataset_params.get('static_categoricals')}")
    print(f"\ntime_varying_known_reals: {dataset_params.get('time_varying_known_reals')}")
    print(f"\ntime_varying_unknown_reals (first 10): {dataset_params.get('time_varying_unknown_reals', [])[:10]}")
else:
    print("No dataset_parameters found on model")
    print("\nModel hparams keys:", list(model.hparams.keys()))

Model's dataset parameters:
max_encoder_length: 60
max_prediction_length: 60
target: close
group_ids: ['group']

time_varying_known_categoricals: []
time_varying_unknown_categoricals: []
static_categoricals: ['group']

time_varying_known_reals: ['hour_sin', 'hour_cos', 'minute_sin', 'minute_cos', 'day_sin']

time_varying_unknown_reals (first 10): ['close', 'open', 'high', 'low', 'volume', 'vwap', 'returns_1m', 'returns_5m', 'returns_15m', 'returns_30m']


In [ ]:
# Cell 6: Create prediction dataset using model's full parameters

# Prepare clean data
df_for_pred = df_recent.copy()

# Clean data - replace inf/nan
numeric_cols = df_for_pred.select_dtypes(include=[np.number]).columns.tolist()
df_for_pred[numeric_cols] = df_for_pred[numeric_cols].replace([np.inf, -np.inf], np.nan)
df_for_pred[numeric_cols] = df_for_pred[numeric_cols].ffill().bfill()

# Clip extreme values
for col in numeric_cols:
    if col not in ['time_idx', 'hour', 'minute', 'day_of_week']:
        q_low, q_high = df_for_pred[col].quantile([0.001, 0.999])
        df_for_pred[col] = df_for_pred[col].clip(lower=q_low, upper=q_high)

# Ensure 'group' matches what model was trained on (likely 'SPY' as string)
df_for_pred['group'] = 'SPY'

# Use TimeSeriesDataSet.from_parameters with ALL model's stored parameters
# This ensures categorical encoders match
prediction_dataset = TimeSeriesDataSet.from_parameters(
    dataset_params,
    df_for_pred,
    predict=True,  # Critical: enable predict mode
)

print(f"Dataset created with {len(prediction_dataset)} samples")

# Create dataloader
pred_dataloader = prediction_dataset.to_dataloader(train=False, batch_size=1, num_workers=0)

# Get predictions
with torch.no_grad():
    for x_batch, y_batch in pred_dataloader:
        pass  # Get the last batch
    
    # Forward pass through model
    output = model(x_batch)

# Extract predictions
raw_pred = output.prediction.squeeze().cpu().numpy()
print(f"Raw prediction shape: {raw_pred.shape}")

# Get point predictions (median if quantiles, otherwise as-is)
if len(raw_pred.shape) == 2:
    pred_array = raw_pred[:, raw_pred.shape[1] // 2]  # Median
else:
    pred_array = raw_pred

# Get reference price
last_close = df_for_pred['close'].iloc[-61]
last_timestamp = df_for_pred['timestamp'].iloc[-61]

print(f"\nLast known price: ${last_close:.2f} at {last_timestamp}")
print(f"Predictions at t+15, t+30, t+45, t+60: ${pred_array[14]:.2f}, ${pred_array[29]:.2f}, ${pred_array[44]:.2f}, ${pred_array[59]:.2f}")

time_varying_known_reals: ['hour_sin', 'hour_cos', 'minute_sin', 'minute_cos', 'day_sin']
time_varying_unknown_reals (70 cols): ['close', 'open', 'high', 'low', 'volume']...

Dataset created with 1 samples


IndexError: index 75 is out of bounds for dimension 1 with size 75